## 06_lenormand_skmob_cz
  Lenormand et al. (2016)-style comparison on CZ NUTS3 with scikit-mobility: unconstrained and
  production-constrained levels, 100 realizations, goodness-of-fit measures.
  Output: shared inputs in data/interim/lenormand_comparison/cz/, metrics CSV. Environment: nuts3-mobility.

In [ ]:
# Shared input for the comparison, CZ part: population per NUTS3 region
import os
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
COUNTRY = "CZ"
YEAR = 2023                                   # same year as notebooks 04 and 05
COL_ID, COL_POP = "nuts3_id", "population"
COL_COUNTRY, COL_YEAR = "country_code", "year"

NUTS3_TABLE = config.PROCESSED_DIR / "nuts3_master_table.parquet"
SHARED_DIR = config.INTERIM_DIR / "lenormand_comparison" / COUNTRY.lower()
POP_CSV = SHARED_DIR / "population.csv"
# ------------------------------------------------------------------

SHARED_DIR.mkdir(parents=True, exist_ok=True)

if POP_CSV.exists():
    pop = pd.read_csv(POP_CSV)
    print("Loaded from checkpoint:", os.path.relpath(POP_CSV), pop.shape)
else:
    # read only the needed columns (no geometry), then filter and sort
    tbl = pd.read_parquet(NUTS3_TABLE, columns=[COL_ID, COL_COUNTRY, COL_YEAR, COL_POP])
    pop = (tbl[(tbl[COL_COUNTRY] == COUNTRY) & (tbl[COL_YEAR] == YEAR)]
           [[COL_ID, COL_POP]]
           .sort_values(COL_ID)
           .reset_index(drop=True))
    if pop[COL_ID].duplicated().any() or pop[COL_POP].isna().any():
        raise ValueError("duplicated region ids or missing population")
    pop.to_csv(POP_CSV, index=False)
    print("Saved:", os.path.relpath(POP_CSV), pop.shape)

print("Regions:", len(pop), "| total population:", int(pop[COL_POP].sum()))
print(pop.head(3).to_string(index=False))

In [ ]:
# PROJ data fix: must run BEFORE geopandas / pyproj are imported
import os
import sys
from pathlib import Path
from importlib.util import find_spec


def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec and spec.submodule_search_locations:  # pip wheel with bundled data
        candidates.append(Path(list(spec.submodule_search_locations)[0]) / "proj_dir" / "share" / "proj")
    for c in candidates:
        if (c / "proj.db").exists():
            return c
    return None


proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)
    print("PROJ data: proj.db found, environment variables set")
else:
    print("PROJ data: proj.db NOT found in the expected places")

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.wkb
from geopandas.array import GeometryArray
from scipy.spatial.distance import pdist, squareform

# ----------------------------- CONFIG -----------------------------
# COUNTRY, YEAR, COL_*, NUTS3_TABLE, SHARED_DIR and pop come from the previous cell
CRS_GEO = "EPSG:4326"
CRS_METRIC = "EPSG:3035"
DIST_CSV = SHARED_DIR / "distance_km.csv"
# ------------------------------------------------------------------

if DIST_CSV.exists():
    dist_km = pd.read_csv(DIST_CSV, index_col=0)
    print("Loaded from checkpoint:", os.path.relpath(DIST_CSV), dist_km.shape)
else:
    # Load the master table; geometry is stored as WKB bytes
    nuts3 = pd.read_parquet(NUTS3_TABLE)

    # Element-wise geometry assignment (vectorized assignment fails with this shapely version)
    geoms = nuts3["geometry"].apply(
        lambda x: shapely.wkb.loads(x) if isinstance(x, (bytes, bytearray)) else x
    ).tolist()
    geom_arr = np.empty(len(geoms), dtype=object)
    for i, g in enumerate(geoms):
        geom_arr[i] = g

    nuts3 = nuts3.drop(columns="geometry")
    nuts3 = gpd.GeoDataFrame(nuts3, geometry=GeometryArray(geom_arr, crs=CRS_GEO))

    # Select the country and year, sorted by region id (same order as the population table)
    cz = (nuts3[(nuts3[COL_COUNTRY] == COUNTRY) & (nuts3[COL_YEAR] == YEAR)]
          .sort_values(COL_ID).reset_index(drop=True))

    # Centroids and pairwise distances in a metric projection
    cz_metric = cz.to_crs(CRS_METRIC)
    coords = np.array([[p.x, p.y] for p in cz_metric.geometry.centroid])
    dist_km = pd.DataFrame(squareform(pdist(coords)) / 1000,   # metres -> km
                           index=cz[COL_ID].to_numpy(), columns=cz[COL_ID].to_numpy())
    dist_km.to_csv(DIST_CSV)
    print("Saved:", os.path.relpath(DIST_CSV), dist_km.shape)

# Checks: same region order as the population table, symmetric, zero diagonal
same_order = (list(dist_km.index) == list(pop[COL_ID])) and (list(dist_km.columns) == list(pop[COL_ID]))
off_diag = dist_km.values[~np.eye(len(dist_km), dtype=bool)]
print("Same region order as population table:", same_order)
print("Symmetric:", bool(np.allclose(dist_km.values, dist_km.values.T)),
      "| zero diagonal:", bool(np.allclose(np.diag(dist_km.values), 0)))
print("Off-diagonal distance (km): min", round(off_diag.min(), 1), "| max", round(off_diag.max(), 1))

In [ ]:
import json
import requests

# ----------------------------- CONFIG -----------------------------
# COUNTRY, COL_ID, SHARED_DIR, pop and config come from the earlier cells
CSV_URL = ("https://csu.gov.cz/docs/107508/8dc6b1a9-6932-6b27-7696-78a2f4d789e8/"
           "sldb2021_vyjizdka_pracoviste_kraj_pohlavi.csv?version=1.0")
CSV_NAME = "sldb2021_vyjizdka_pracoviste_kraj_pohlavi.csv"
RAW_COMMUTE_DIR = config.RAW_DIR / "commuting" / "cz"
GISCO_GEOJSON = config.RAW_DIR / "nuts3" / "NUTS_RG_03M_2024_4326.geojson"

# original Czech column names stay unchanged
COL_ORIGIN_LEVEL, COL_DEST_LEVEL = "uzemi_cis", "kraj_dojizdky_cis"
COL_DEST_CODE, COL_SEX = "kraj_dojizdky_kod", "pohlavi_kod"
COL_ORIGIN_NAME, COL_DEST_NAME, COL_VALUE = "uzemi_txt", "kraj_dojizdky_txt", "hodnota"
COL_ACTIVITY, COL_INDICATOR = "aktivita_txt", "ukaz_txt"
KRAJ_LEVEL_CODE = 100                 # code of the kraj level in the code list

GISCO_COL_ID, GISCO_COL_NAME, GISCO_COL_LEVEL = "NUTS_ID", "NUTS_NAME", "LEVL_CODE"
NUTS_LEVEL = 3
OD_CSV = SHARED_DIR / "od_observed.csv"
# ------------------------------------------------------------------

ids = pop[COL_ID].tolist()            # region order shared by all inputs

if OD_CSV.exists():
    od = pd.read_csv(OD_CSV, index_col=0)
    print("Loaded from checkpoint:", os.path.relpath(OD_CSV), od.shape)
else:
    # 1) idempotent download
    RAW_COMMUTE_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = RAW_COMMUTE_DIR / CSV_NAME
    if csv_path.exists():
        print("CSV already present, download skipped:", os.path.relpath(csv_path))
    else:
        try:
            response = requests.get(CSV_URL, timeout=120)
            response.raise_for_status()
            csv_path.write_bytes(response.content)
            print("Downloaded:", os.path.relpath(csv_path))
        except Exception as e:
            print("Download failed:", type(e).__name__)
            print("Manual fix: open CSV_URL (config of this cell) in a browser and save the file as")
            print("  ", CSV_NAME, "into the folder:", os.path.relpath(RAW_COMMUTE_DIR))
            raise FileNotFoundError(CSV_NAME) from None
    raw = pd.read_csv(csv_path)

    # 2) kraj -> kraj rows, both sexes combined
    mask = ((raw[COL_ORIGIN_LEVEL] == KRAJ_LEVEL_CODE)
            & (raw[COL_DEST_LEVEL] == KRAJ_LEVEL_CODE)
            & raw[COL_DEST_CODE].notna()
            & raw[COL_SEX].isna())
    kk = raw.loc[mask].copy()
    dup_pairs = int(kk.duplicated(subset=[COL_ORIGIN_NAME, COL_DEST_NAME]).sum())

    # 3) pivot to an origin x destination matrix (missing pairs -> 0)
    od_by_name = kk.pivot(index=COL_ORIGIN_NAME, columns=COL_DEST_NAME, values=COL_VALUE)
    n_missing_cells = int(od_by_name.isna().sum().sum())
    od_by_name = od_by_name.fillna(0)
    same_names = set(od_by_name.index) == set(od_by_name.columns)
    if not same_names:
        raise ValueError("origin and destination kraj names differ")
    od_by_name = od_by_name.reindex(columns=od_by_name.index)

    # 4) kraj names -> NUTS3 codes (read only the GISCO properties, no geometry)
    with open(GISCO_GEOJSON, encoding="utf-8") as f:
        props = pd.DataFrame([ft["properties"] for ft in json.load(f)["features"]])
    props = props[props[GISCO_COL_ID].str.startswith(COUNTRY) & (props[GISCO_COL_LEVEL] == NUTS_LEVEL)]
    name_to_code = dict(zip(props[GISCO_COL_NAME], props[GISCO_COL_ID]))
    unmatched = sorted(set(od_by_name.index) - set(name_to_code))
    if unmatched:
        raise ValueError("unmatched kraj names: " + ", ".join(unmatched))

    od = od_by_name.rename(index=name_to_code, columns=name_to_code)
    if not (set(od.index) == set(ids) == set(od.columns)):
        raise ValueError("region codes differ from the population table")
    od = od.loc[ids, ids]
    od.to_csv(OD_CSV)

    print("Filtered rows:", len(kk), "| duplicated pairs:", dup_pairs,
          "| cells filled with 0:", n_missing_cells)
    print("Activity values:", kk[COL_ACTIVITY].unique().tolist())
    print("Indicator values:", kk[COL_INDICATOR].unique().tolist())
    print("Saved:", os.path.relpath(OD_CSV))

# checks
print("OD matrix shape:", od.shape, "| same order as population:",
      list(od.index) == ids and list(od.columns) == ids)
print("Total commuters (all cells):", int(od.values.sum()))
print("Within-kraj share (diagonal):", round(float(np.trace(od.values) / od.values.sum()), 3))

In [ ]:
import shapely.wkb
from geopandas.array import GeometryArray
import skmob

# ----------------------------- CONFIG -----------------------------
# COUNTRY, YEAR, COL_*, NUTS3_TABLE, CRS_GEO, pop and od come from the earlier cells
TILE_ID = "tile_ID"
COL_FLOW_ORIGIN, COL_FLOW_DEST, COL_FLOW = "origin", "destination", "flow"
# ------------------------------------------------------------------

ids = pop[COL_ID].tolist()

# Load geometry again (the shared CSVs have none); element-wise assignment because
# vectorized assignment fails with this shapely version
nuts3 = pd.read_parquet(NUTS3_TABLE)
geoms = nuts3["geometry"].apply(
    lambda x: shapely.wkb.loads(x) if isinstance(x, (bytes, bytearray)) else x
).tolist()
geom_arr = np.empty(len(geoms), dtype=object)
for i, g in enumerate(geoms):
    geom_arr[i] = g
nuts3 = gpd.GeoDataFrame(nuts3.drop(columns="geometry"),
                         geometry=GeometryArray(geom_arr, crs=CRS_GEO))

# Region table in the shared order (sorted by region id)
cz = (nuts3[(nuts3[COL_COUNTRY] == COUNTRY) & (nuts3[COL_YEAR] == YEAR)]
      .sort_values(COL_ID).reset_index(drop=True))
assert cz[COL_ID].tolist() == ids, "region order differs from the population table"

# Tessellation: one row per region (id, geometry, population)
tessellation = cz[[COL_ID, "geometry", COL_POP]].rename(columns={COL_ID: TILE_ID})
tessellation = gpd.GeoDataFrame(tessellation, geometry="geometry", crs=CRS_GEO)

# Flows in long format: one row per origin-destination pair (diagonal included).
# Axis names are set explicitly because rows and columns carried the same name.
flows_long = (
    od
    .rename_axis(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST)
    .stack()
    .rename(COL_FLOW)
    .reset_index()
)

fdf = skmob.FlowDataFrame(
    flows_long,
    origin=COL_FLOW_ORIGIN,
    destination=COL_FLOW_DEST,
    flow=COL_FLOW,
    tessellation=tessellation,
    tile_id=TILE_ID,
)

n_regions = len(ids)
print("Tiles in tessellation:", len(tessellation))
print("Flow rows:", len(fdf), "| expected:", n_regions ** 2)
print("Total commuters in fdf:", int(fdf[COL_FLOW].sum()), "| OD matrix:", int(od.values.sum()))

In [ ]:
# ----------------------------- CONFIG -----------------------------
OUTFLOW_COL = "tot_outflow"
# ------------------------------------------------------------------

# Region order must be identical in the tessellation and in the OD matrix
assert tessellation[TILE_ID].tolist() == list(od.index) == list(od.columns)

# Observed flows without the diagonal (Lenormand et al.: T_ii = 0)
observed_offdiag = od.to_numpy(dtype=float).copy()
np.fill_diagonal(observed_offdiag, 0)

# Outflow per region without self-loops (generate() never creates self-loops)
tessellation[OUTFLOW_COL] = observed_offdiag.sum(axis=1)

total_offdiag = int(observed_offdiag.sum())
print("Total off-diagonal commuters:", total_offdiag)
print("Sum of outflows equals off-diagonal total:", int(tessellation[OUTFLOW_COL].sum()) == total_offdiag)
print(tessellation[[TILE_ID, OUTFLOW_COL]].head(3).to_string(index=False))

In [ ]:
import warnings
from skmob.models.gravity import Gravity

# ----------------------------- CONFIG -----------------------------
GRAVITY_LAWS = {"gravity_power": "power_law", "gravity_exp": "exponential"}
CONSTRAINT_LEVELS = {"UM": "globally constrained", "PCM": "singly constrained"}
RESULTS_DIR = config.PROCESSED_DIR / "06_lenormand_skmob_cz"
GRAVITY_PARAMS_CSV = RESULTS_DIR / "gravity_params.csv"
# ------------------------------------------------------------------

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Fit on the off-diagonal flows only (T_ii = 0, as in the paper)
flows_off = flows_long[flows_long[COL_FLOW_ORIGIN] != flows_long[COL_FLOW_DEST]]
fdf_off = skmob.FlowDataFrame(
    flows_off,
    origin=COL_FLOW_ORIGIN,
    destination=COL_FLOW_DEST,
    flow=COL_FLOW,
    tessellation=tessellation,
    tile_id=TILE_ID,
)

if GRAVITY_PARAMS_CSV.exists():
    gravity_params = pd.read_csv(GRAVITY_PARAMS_CSV)
    print("Loaded from checkpoint:", os.path.relpath(GRAVITY_PARAMS_CSV))
else:
    rows = []
    for law, deterrence in GRAVITY_LAWS.items():
        for level, gravity_type in CONSTRAINT_LEVELS.items():
            # Record warnings instead of silencing them; only the class names are kept
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                model = Gravity(gravity_type=gravity_type, deterrence_func_type=deterrence)
                model.fit(fdf_off, relevance_column=COL_POP)
            rows.append({
                "model": f"{law}_{level}",
                "deterrence_func_type": model.deterrence_func_type,
                "gamma": model.deterrence_func_args[0],
                "origin_exp": model.origin_exp,
                "destination_exp": model.destination_exp,
                "gravity_type": model.gravity_type,
                "fit_warnings": ";".join(sorted({w.category.__name__ for w in caught})),
            })
    gravity_params = pd.DataFrame(rows)
    gravity_params.to_csv(GRAVITY_PARAMS_CSV, index=False)
    print("Saved:", os.path.relpath(GRAVITY_PARAMS_CSV))

# Build the model objects from the saved parameters (works with or without a fresh fit)
gravity_models = {
    r.model: Gravity(deterrence_func_type=r.deterrence_func_type,
                     deterrence_func_args=[r.gamma],
                     origin_exp=r.origin_exp,
                     destination_exp=r.destination_exp,
                     gravity_type=r.gravity_type)
    for r in gravity_params.itertuples()
}

print(gravity_params.to_string(index=False))

In [ ]:
from skmob.models.radiation import Radiation

# ----------------------------- CONFIG -----------------------------
N_REPS = 100                        # stochastic realizations per model (as in the paper)
SEED_BASE = 0                       # realization i uses seed SEED_BASE + i
MODEL_OBJECTS = {**gravity_models, "radiation_PCM": Radiation()}
MODELS = list(MODEL_OBJECTS)
REALIZATION_PATH = lambda name: RESULTS_DIR / f"realizations_{name}.npy"
# ------------------------------------------------------------------

realizations = {}

for name in MODELS:
    path = REALIZATION_PATH(name)

    if path.exists():
        realizations[name] = np.load(path)
        print(name, "- loaded from checkpoint:", os.path.relpath(path))
        continue

    model = MODEL_OBJECTS[name]
    mats = []
    warning_types = set()
    for i in range(N_REPS):
        np.random.seed(SEED_BASE + i)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            synth = model.generate(
                tessellation,
                tile_id_column=TILE_ID,
                tot_outflows_column=OUTFLOW_COL,
                relevance_column=COL_POP,
                out_format="flows",
            )
        warning_types |= {w.category.__name__ for w in caught}

        # FlowDataFrame -> plain DataFrame -> matrix in the shared region order
        mat = (
            pd.DataFrame(synth)
            .pivot(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST, values=COL_FLOW)
            .reindex(index=ids, columns=ids)
            .fillna(0)
            .to_numpy()
        )
        mats.append(mat)

    realizations[name] = np.stack(mats)          # shape: (N_REPS, n_regions, n_regions)
    np.save(path, realizations[name])
    print(name, "- generated and saved:", os.path.relpath(path),
          "| warnings:", ";".join(sorted(warning_types)) or "none")

print()
for name in MODELS:
    arr = realizations[name]
    diag_sum = float(np.trace(arr, axis1=1, axis2=2).sum())
    mean_total = float(arr.sum(axis=(1, 2)).mean())
    print(f"{name}: shape {arr.shape} | diagonal sum {diag_sum:.0f} | "
          f"mean total {mean_total:.0f} (observed off-diagonal {observed_offdiag.sum():.0f})")

In [ ]:
import skmob.measures.evaluation as ev

# ----------------------------- CONFIG -----------------------------
METRICS_BY_REP_CSV = RESULTS_DIR / "metrics_skmob_by_rep.csv"
METRICS_SUMMARY_CSV = RESULTS_DIR / "metrics_skmob_summary.csv"
# ------------------------------------------------------------------

# Off-diagonal pairs only (182 pairs for 14 regions)
offdiag = ~np.eye(len(ids), dtype=bool)
obs_flat = observed_offdiag[offdiag]

if METRICS_BY_REP_CSV.exists():
    by_rep = pd.read_csv(METRICS_BY_REP_CSV)
    print("Loaded from checkpoint:", os.path.relpath(METRICS_BY_REP_CSV), by_rep.shape)
else:
    rows = []
    for name in MODELS:
        for rep, mat in enumerate(realizations[name]):
            sim_flat = mat[offdiag].astype(float)
            # information gain is infinite when a real pair gets zero predicted flow
            with np.errstate(divide="ignore", invalid="ignore"):
                ig = ev.information_gain(obs_flat, sim_flat)
            rows.append({
                "model": name,
                "rep": rep,
                "cpc": ev.common_part_of_commuters(obs_flat, sim_flat),
                "information_gain": ig,
                "zero_pairs": int(((obs_flat > 0) & (sim_flat == 0)).sum()),
            })
    by_rep = pd.DataFrame(rows)
    by_rep.to_csv(METRICS_BY_REP_CSV, index=False)
    print("Saved:", os.path.relpath(METRICS_BY_REP_CSV), by_rep.shape)

# Summary per model over the realizations
summary = (
    by_rep.groupby("model", sort=False)
    .agg(cpc_mean=("cpc", "mean"), cpc_std=("cpc", "std"),
         cpc_min=("cpc", "min"), cpc_max=("cpc", "max"),
         zero_pairs_mean=("zero_pairs", "mean"),
         n_ig_inf=("information_gain", lambda s: int((~np.isfinite(s)).sum())),
         ig_mean_finite=("information_gain", lambda s: s[np.isfinite(s)].mean()))
    .reindex(MODELS)
)
summary.to_csv(METRICS_SUMMARY_CSV)
print("Saved:", os.path.relpath(METRICS_SUMMARY_CSV))
print(summary.round(4).to_string())

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
FIG_DIRS = [Path("../figures"), Path("../plots")]
FIG_NAME = "cz_lenormand_cpc_skmob.png"
LEVEL_ORDER = ["UM", "PCM"]                       # plotted bottom to top
LEVEL_LABELS = {"UM": "UM\nunconstrained", "PCM": "PCM\nproduction constrained"}
# law key (as in the model names) -> label, color (categorical slots 1-3), marker, vertical offset
LAWS = {
    "gravity_exp":   ("Gravity, exponential (NGrav exp)", "#2a78d6", "o",  0.20),
    "gravity_power": ("Gravity, power law (NGrav pow)",   "#eb6834", "s",  0.00),
    "radiation":     ("Radiation (original)",              "#1baf7a", "^", -0.20),
}
XLIM = (0.5, 0.9)
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
# ------------------------------------------------------------------

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 3.6))
for name, row in summary.iterrows():
    law, level = name.rsplit("_", 1)
    label, color, marker, dy = LAWS[law]
    y = LEVEL_ORDER.index(level) + dy
    # marker with a surface-colored ring so overlapping points stay readable
    ax.scatter(row["cpc_mean"], y, s=90, marker=marker, color=color,
               edgecolor="#fcfcfb", linewidth=1.5, zorder=3,
               label=label if level == "PCM" else None)
    ax.text(row["cpc_mean"] + 0.008, y, f"{row['cpc_mean']:.3f}",
            va="center", ha="left", fontsize=9, color=INK)

ax.set_yticks(range(len(LEVEL_ORDER)))
ax.set_yticklabels([LEVEL_LABELS[l] for l in LEVEL_ORDER], color=INK)
ax.set_ylim(-0.6, len(LEVEL_ORDER) - 0.4)
ax.set_xlim(*XLIM)
ax.set_xlabel(f"Common part of commuters (CPC); axis starts at {XLIM[0]}", color=INK)
ax.set_title("CZ NUTS3: CPC by law and constraint level (skmob, mean of "
             f"{N_REPS} realizations)", color=INK, fontsize=11, loc="left")
ax.grid(axis="x", color=GRID, linewidth=0.8, zorder=0)
ax.tick_params(colors=MUTED)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=9)

fig.tight_layout()
for d in FIG_DIRS:
    fig.savefig(d / FIG_NAME, dpi=200, bbox_inches="tight")
    print("Saved:", os.path.relpath(d / FIG_NAME))
plt.show()